In [12]:
import os
from PIL import Image

def compress_images_in_folder(folder_path, max_size_bytes=1*1024*1024, quality_step=5):
    """
    Batch compress images in a folder that exceed max_size_bytes to within max_size_bytes,
    minimizing quality loss as much as possible.
    Supports JPEG and PNG formats.
    """
    compressed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if not os.path.isfile(file_path):
            continue
        if os.path.getsize(file_path) <= max_size_bytes:
            skipped_count += 1
            continue
        try:
            with Image.open(file_path) as img:
                # JPEG compression
                if img.format == 'JPEG':
                    quality = 95
                    while quality > 10:
                        img.save(file_path, 'JPEG', quality=quality, optimize=True)
                        if os.path.getsize(file_path) <= max_size_bytes:
                            compressed_count += 1
                            break
                        quality -= quality_step
                # PNG compression
                elif img.format == 'PNG':
                    img.save(file_path, 'PNG', optimize=True)
                    if os.path.getsize(file_path) > max_size_bytes:
                        # PNG compression is limited, try converting to JPEG
                        rgb_img = img.convert('RGB')
                        new_file_path = file_path.rsplit('.', 1)[0] + '.jpg'
                        quality = 95
                        while quality > 10:
                            rgb_img.save(new_file_path, 'JPEG', quality=quality, optimize=True)
                            if os.path.getsize(new_file_path) <= max_size_bytes:
                                compressed_count += 1
                                break
                            quality -= quality_step
                        os.remove(file_path)
                    else:
                        compressed_count += 1
                else:
                    skipped_count += 1
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")
            skipped_count += 1

    print(f"Compression finished: {compressed_count} files compressed, {skipped_count} files skipped.")

# Example usage
compress_images_in_folder('../images/gallery/')

Compression finished: 1 files compressed, 40 files skipped.


In [13]:
import shutil
def generate_thumbnails(src_folder, dst_folder, thumb_size=(400, 225)):
    """
    Batch generate thumbnails for images in src_folder, keeping aspect ratio,
    with max size thumb_size. Unsupported formats are copied directly.
    Skip if thumbnail already exists.
    """
    os.makedirs(dst_folder, exist_ok=True)
    count = 0
    skipped = 0
    for fname in os.listdir(src_folder):
        src_path = os.path.join(src_folder, fname)
        dst_path = os.path.join(dst_folder, fname)
        if not os.path.isfile(src_path):
            continue
        if os.path.exists(dst_path):
            skipped += 1
            continue
        try:
            with Image.open(src_path) as img:
                img_format = img.format
                # Supported formats: generate thumbnail
                if img_format in ['JPEG', 'PNG', 'WEBP', 'BMP']:
                    img.thumbnail(thumb_size)
                    img.save(dst_path, format=img_format, quality=80, optimize=True)
                else:
                    shutil.copy2(src_path, dst_path)
                count += 1
        except Exception as e:
            print(f'Skipped {src_path}: {e}')
    print(f'Generated {count} thumbnails in {dst_folder}, {skipped} files skipped (already exist).')

# Example usage
generate_thumbnails('../images/gallery/', '../images/gallery_thumbs/')

Generated 1 thumbnails in ../images/gallery_thumbs/, 40 files skipped (already exist).
